In [1]:
import pandas as pd

from src import PROJECT_DIR, logging
from src import utils as src_utils

from discovery_utils.getters import crunchbase
from discovery_utils.utils import (
    analysis_crunchbase,
    analysis,
    charts,
    google,
    google_slides,
)

PROJECT_NAME = src_utils.PROJECT_NAME
OUTPUT_DIR = src_utils.OUTPUT_DIR / "mission_radar"

import importlib;
importlib.reload(src_utils);
importlib.reload(charts)
importlib.reload(google);
importlib.reload(google_slides);

In [2]:
CB = crunchbase.CrunchbaseGetter()

2025-04-29 11:46:44,795 - discovery_utils.getters.crunchbase - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2025-04-29 11:46:44,947 - discovery_utils.getters.crunchbase - INFO - Latest Crunchbase version found: Crunchbase_2025-04-28


In [3]:
from discovery_utils.utils.io import remap_dict
investment_stages = {'early_stage': ['pre_seed',
  'seed',
  'angel',
  'series_a',
  'series_b',
  'convertible_note',
  'equity_crowdfunding',
  'product_crowdfunding',
  'non_equity_assistance',
  'initial_coin_offering'],
 'growth_stage': ['series_c',
  'series_d',
  'series_e',
  'series_f',
  'series_g',
  'series_h',
  'series_i',
  'series_j'],
 'late_stage': ['private_equity',
  'post_ipo_equity',
  'post_ipo_debt',
  'post_ipo_secondary',
  'secondary_market'],
 'other': ['corporate_round', 'debt_financing', 'grant', 'series_unknown', 'undisclosed'],
#  'uncategorized': ['series_unknown', 'undisclosed']}
}
investment_type_to_stage = remap_dict(investment_stages)

In [4]:
def get_ids_from_config(config_name):
    config = src_utils.get_config_dict(config_name)
    selected_df = src_utils.get_companies_from_config(CB, config)

    selected_texts_df = CB.get_organisation_text(selected_df)

    # relevant_check_df = pd.read_json(src_utils.OUTPUT_DIR / f"llm_check_MS_{config_name}.jsonl", lines=True)
    relevant_check_df = pd.read_json(src_utils.OUTPUT_DIR / f"llm_check_v2_{config_name}.jsonl", lines=True)

    relevant_checked_df = (
        selected_df
        .merge(relevant_check_df[['id', 'is_relevant']], left_on='id', right_on='id', how='left')
        .merge(selected_texts_df[['id', 'text']], left_on='id', right_on='id', how='left')
    )
    matching_ids = relevant_checked_df.query("is_relevant == 'yes'").id.tolist()

    return matching_ids

low_carbon_heating_configs = [
    "biomass_heating",    
    "district_heating",
    "geothermal_energy",
    "heat_pumps",
    "heat_storage",
    "hydrogen_heating",
    "micro_chp",
    "solar_thermal",
]
lch_ids = []
for config in low_carbon_heating_configs:
    lch_ids.extend(get_ids_from_config(config))
lch_ids = list(set(lch_ids))

2025-04-29 11:46:56,844 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/enriched/organizations_full.parquet
2025-04-29 11:46:57,059 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-29 11:46:57,071 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-29 11:46:57,094 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-29 11:46:57,095 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-29 11:46:57,099 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', '

In [5]:
FUNDING_ROUND_TYPES = ["angel", "pre_seed", "seed", "series_a", "series_b"]

PRESENT_QUARTER = "2025-Q1"

In [6]:
import datetime
# import datetime data type

def date_to_quarter(date: datetime.datetime) -> str:
    return f"{date.year}-Q{date.quarter}"

def get_present_quarter():
    today = datetime.date.today()
    return f"{today.year}-Q{int((today.month - 1) / 3) + 1}"

In [7]:
import altair as alt
from discovery_utils.utils.charts import configure_plots
from discovery_utils.utils.charts import configure_titles

def aggregate_by_funding_round_types(funding_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate investment data by funding round type and year

    Returns a dataframe with the following columns:
        - 'year': Year of the funding round
        - 'investment_type': Type of the funding round
        - 'raised_amount_gbp': Total amount raised in GBP
        - 'counts': Number of funding rounds
    """
    return (
        funding_df.drop_duplicates(["funding_round_id"])
        .groupby(["year", "investment_type"])
        .agg(raised_amount_gbp=("raised_amount_gbp", "sum"), counts=("funding_round_id", "count"))
        .reset_index()
    )

def chart_investment_types_counts(funding_round_types_df: pd.DataFrame, title: str = "", time_period_col: str="year") -> alt.Chart:
    """Create a bar chart of the number of funding rounds by types"""
    fig = _chart_investment_types(
        funding_round_types_df,
        value_column="counts",
        value_label="Number of funding rounds",
        time_period_col=time_period_col,
    )
    fig = configure_titles(fig, title)
    return fig

def chart_investment_types(funding_round_types_df: pd.DataFrame, title: str = "", time_period_col: str="year") -> alt.Chart:
    """Create a bar chart of investment amounts by type"""
    fig = _chart_investment_types(
        funding_round_types_df.assign(raised_amount_gbp=lambda df: df.raised_amount_gbp / 1e3),
        value_column="raised_amount_gbp",
        value_label="Raised amount (£ millions)",
        time_period_col=time_period_col,
    )
    fig = configure_titles(fig, title)
    return fig

def _chart_investment_types(
    funding_round_types_df: pd.DataFrame,
    value_column: str = "raised_amount_gbp",
    value_label: str = "Raised amount (£ millions)",
    colour_column: str = "investment_type",
    colour_label: str = "Investment type",
    time_period_col: str = "year",
    stack_order: str = None,
) -> alt.Chart:
    """Create a bar chart of investment types"""
    if stack_order is None:
        stack_order = colour_column
    fig = (
        alt.Chart(
            funding_round_types_df,
            width=500,
            height=300,
        )
        .mark_bar()
        .encode(
            x=alt.X(f"{time_period_col}:O", title=""),
            y=alt.Y(f"{value_column}:Q", title=value_label),
            color=alt.Color(
                f"{colour_column}:N",
                title=colour_label,
            ),
            order=alt.Order(stack_order),
            tooltip=[time_period_col, colour_column, value_column],
        )
    )
    return configure_plots(fig)

def aggregate_by_funding_round_types2(funding_rounds_df: pd.DataFrame, min_year: int, max_year: int, period: str="year") -> pd.DataFrame:
    """Aggregate investment data by funding round type and year"""
    period = period[0].capitalize()
    grouped = (
        funding_rounds_df
        .drop_duplicates(["funding_round_id"])
        # .query("investment_stage != 'early_stage' and year != 2015")
        .assign(time_period = lambda df: pd.to_datetime(df.announced_on))
        .assign(time_period = lambda df: df.time_period.dt.to_period(period))
        .groupby(["time_period", "investment_type"])
        .agg(
            counts=("funding_round_id", "count"),
            raised_amount_usd=("raised_amount_usd", "sum"),
            raised_amount_gbp=("raised_amount_gbp", "sum"),
        )
        .reset_index()
        .assign(time_period = lambda df: df.time_period.astype("datetime64[ns]"))
        .pipe(analysis.impute_empty_periods, "time_period", period=period, min_year=min_year, max_year=max_year)
        .replace({"investment_type": {0: "other"}})
    )
    if period == "Y":
        return grouped.assign(year = lambda df: df.time_period.dt.year).drop(columns="time_period")
    if period == "Q":
        return grouped.assign(quarter = lambda df: df.time_period.apply(date_to_quarter)).query("time_period <= @PRESENT_QUARTER")

In [8]:
# def produce_stats(CB, matching_ids: list[str], category_name: str, funding_round_types=FUNDING_ROUND_TYPES, prefix: str=None) -> None:
#     """ Produce stats for the companies and output charts """ 
#     # Check companies by querying ids
#     matchings_orgs_df = CB.organisations_enriched.query("id in @matching_ids")

#     # Get the funding rounds for the matching companies
#     funding_rounds_df = (
#         CB.select_funding_rounds(org_ids=matching_ids, funding_round_types=funding_round_types)
#     )

#     # organise investors by each funding round
#     investors_df = (
#         CB.funding_rounds_enriched
#         .query("funding_round_id in @funding_rounds_df.funding_round_id")
#         .groupby("funding_round_id")
#         .agg(investor_name=("investor_name", list))
#         .reset_index()
#     )

#     funding_rounds_df = (
#         funding_rounds_df
#         .drop(columns=["investor_name"])
#         .merge(investors_df, on="funding_round_id", how="left")
#         .assign(investment_stage = lambda df: df.investment_type.map(investment_type_to_stage, na_action=None))
#     )

#     # generate basic time series
#     ts_df = analysis_crunchbase.get_timeseries(
#         matchings_orgs_df, 
#         funding_rounds_df, 
#         period='year', 
#         min_year=2014, 
#         max_year=2025
#     )
#     growth_rates = analysis.smoothed_growth(ts_df, year_start=2020, year_end=2024)
#     growth_rates_df = pd.DataFrame(growth_rates, columns=[category_name]).T.reset_index().rename(columns={'index': 'theme'})  
#     growth_magnitude_df = (
#         analysis.magnitude_growth(ts_df, year_start=2020, year_end=2024)
#         .assign(theme=category_name)
#         .reset_index()
#         .rename(columns={'index': 'variable'})
#     )

#     # generate basic time series
#     ts_quarterly_df = (
#         analysis_crunchbase.get_timeseries(
#             matchings_orgs_df, 
#             funding_rounds_df, 
#             period='quarter', 
#             min_year=2023, 
#             max_year=2025
#         )
#         .assign(quarter = lambda df: df.time_period.apply(date_to_quarter))
#         .query("quarter <= @get_present_quarter()")
#     )

#     # Let's look into breakdown of deal types
#     # deals_df, deal_counts_df = analysis_crunchbase.get_funding_by_year_and_range(funding_rounds_df, 2014, 2025)
#     # aggregated_funding_types_df = analysis_crunchbase.aggregate_by_funding_round_types(funding_rounds_df)
#     # investment_types_fig = analysis_crunchbase.chart_investment_types(aggregated_funding_types_df)
    
#     # IPOs and acquisitions
#     ipos_df = CB.ipos.query("org_id in @matching_ids")
#     acquisitions_df = CB.acquisitions.query("acquiree_id in @matching_ids")

#     if len(ipos_df) > 0:
#         ipos_df.to_csv(OUTPUT_DIR / f"ipos_{category_name}.csv", index=False)

#     if len(acquisitions_df) > 0:
#         acquisitions_df.to_csv(OUTPUT_DIR / f"acquisitions_{category_name}.csv", index=False)
        

#     # Figure variables
#     if prefix is None:
#         prefix = f"{OUTPUT_DIR}/charts/{category_name}_"
#     _scale = 2

#     # Chart: Investment amounts (total investment, quarterly)
#     fig = charts.ts_bar(
#         ts_quarterly_df,
#         variable='raised_amount_gbp_total',
#         variable_title="Raised amount, £ millions",
#         category_column="_category",
#         time_column="quarter"
#     )
#     fig = charts.configure_plots(fig, chart_title=f"Funding raised over time for {category_name}")
#     chart_filename = f"{prefix}raised_amount_quarterly.png"
#     fig.save(chart_filename, scale_factor=_scale)

#     fig = charts.ts_bar(
#         ts_df,
#         variable='raised_amount_gbp_total',
#         variable_title="Raised amount, £ millions",
#         category_column="_category",
#     )
#     fig = charts.configure_plots(fig, chart_title=f"Funding raised over time for {category_name}")
#     chart_filename = f"{prefix}raised_amount_quarterly.png"
#     fig.save(chart_filename, scale_factor=_scale)

#     # Number of companies
#     # fig = charts.ts_bar(
#     #     ts_df,
#     #     variable='n_orgs_founded',
#     #     variable_title="Number of companies founded",
#     #     category_column="_category",
#     # )
#     # fig = charts.configure_plots(fig, chart_title=f"Number of founded {category_name} companies")
#     # chart_filename = f"{prefix}no_of_companies.png"
#     # fig.save(chart_filename, scale_factor=_scale)    

#     # Investment amounts (by type)
#     # investment_types_fig = analysis_crunchbase.chart_investment_types(aggregated_funding_types_df)
#     # investment_types_fig = charts.configure_plots(investment_types_fig, chart_title=f"Breakdown of investment types for {category_name}")
#     # investment_types_chart_filename = f"{prefix}investment_types.png"
#     # investment_types_fig.save(investment_types_chart_filename, scale_factor=_scale)

#     # # Investment counts (by type)
#     # investment_types_counts_fig = analysis_crunchbase.chart_investment_types_counts(aggregated_funding_types_df)
#     # investment_types_counts_fig = charts.configure_plots(investment_types_counts_fig, chart_title=f"Number of investments by type for {category_name}")
#     # investment_types_counts_chart_filename = f"{prefix}investment_types_counts.png"
#     # investment_types_counts_fig.save(investment_types_counts_chart_filename, scale_factor=_scale)

#     return ts_quarterly_df, growth_magnitude_df, ipos_df, acquisitions_df, matchings_orgs_df, funding_rounds_df


In [9]:
get_present_quarter()

'2025-Q2'

In [10]:
def produce_stats(CB, matching_ids: list[str], category_name: str, funding_round_types=FUNDING_ROUND_TYPES, prefix: str=None) -> None:
# matching_ids = lch_ids
# category_name = "Low carbon heating_test"
# funding_round_types=None
# prefix = f"{OUTPUT_DIR}/charts/test_{category_name}_"
    """ Produce stats for the companies and output charts """ 

    present_quarter = PRESENT_QUARTER
    
    # Check companies by querying ids
    matchings_orgs_df = CB.organisations_enriched.query("id in @matching_ids")

    # Get the funding rounds for the matching companies
    funding_rounds_df = (
        CB.select_funding_rounds(org_ids=matching_ids, funding_round_types=funding_round_types)
    )

    # organise investors by each funding round
    investors_df = (
        CB.funding_rounds_enriched
        .query("funding_round_id in @funding_rounds_df.funding_round_id")
        .groupby("funding_round_id")
        .agg(investor_name=("investor_name", list))
        .reset_index()
    )

    funding_rounds_df = (
        funding_rounds_df
        .drop(columns=["investor_name"])
        .merge(investors_df, on="funding_round_id", how="left")
        .assign(investment_stage = lambda df: df.investment_type.map(investment_type_to_stage, na_action=None))
    )

    # Overall investment time series, yearly
    ts_df = analysis_crunchbase.get_timeseries(
        matchings_orgs_df, 
        funding_rounds_df, 
        period='year', 
        min_year=2014, 
        max_year=2025
    )

    # Investment for early and growth stage
    funding_rounds_startup_df = funding_rounds_df.query("investment_stage in ['early_stage', 'growth_stage']")
    ts_startup_df = analysis_crunchbase.get_timeseries(
        matchings_orgs_df, 
        funding_rounds_startup_df, 
        period='year', 
        min_year=2014, 
        max_year=2025
    )

    growth_rates = analysis.smoothed_growth(ts_startup_df, year_start=2020, year_end=2024)
    growth_rates_df = pd.DataFrame(growth_rates, columns=[category_name]).T.reset_index().rename(columns={'index': 'theme'})  
    growth_magnitude_df = (
        analysis.magnitude_growth(ts_startup_df, year_start=2020, year_end=2024)
        .assign(theme=category_name)
        .reset_index()
        .rename(columns={'index': 'variable'})
    )

    #####
    # Overall investment time series, quarterly
    ts_quarterly_df = (
        analysis_crunchbase.get_timeseries(
            matchings_orgs_df, 
            funding_rounds_df, 
            period='quarter', 
            min_year=2023, 
            max_year=2025
        )
        .assign(quarter = lambda df: df.time_period.apply(date_to_quarter))
        .query("quarter <= @present_quarter")
    )

    ts_quarterly_startup_df = (
        analysis_crunchbase.get_timeseries(
            matchings_orgs_df, 
            funding_rounds_startup_df, 
            period='quarter', 
            min_year=2023, 
            max_year=2025
        )
        .assign(quarter = lambda df: df.time_period.apply(date_to_quarter))
        .query("quarter <= @present_quarter")
    )

    
    previous_four_quarters = ts_quarterly_startup_df.query("quarter < @present_quarter").sort_values("quarter").tail(4).quarter.tolist()
    previous_four_quarters_mean_df = (
        ts_quarterly_startup_df
        .query("quarter in @previous_four_quarters")
        .assign(_col = "previous_four_quarters")
        .groupby("_col")
        .agg(
            raised_amount_gbp_total=("raised_amount_gbp_total", "mean"),
            n_rounds=("n_rounds", "mean"),
            n_orgs_founded = ("n_orgs_founded", "mean")
        )
        .T
        .reset_index()
        .rename(columns={"index": "variable"})
    )

    present_quarter_df = (
        ts_quarterly_startup_df.query("quarter == @present_quarter")
        # rename index to "present_quarter"
        .assign(_col = "magnitude")
        .groupby("_col")
        .agg(
            raised_amount_gbp_total=("raised_amount_gbp_total", "mean"),
            n_rounds=("n_rounds", "mean"),
            n_orgs_founded = ("n_orgs_founded", "mean")
        )
        .T.reset_index().rename(columns={"index": "variable"})
        )

    growth_magnitude_quarterly_df = (
        previous_four_quarters_mean_df
        .merge(present_quarter_df, on="variable", how="left")
        .assign(growth = lambda df: (df.magnitude - df.previous_four_quarters) / df.previous_four_quarters * 100)
        .assign(theme=category_name)
    )
    ####

    # Let's look into breakdown of deal types
    # deals_df, deal_counts_df = analysis_crunchbase.get_funding_by_year_and_range(funding_rounds_df, 2014, 2025)
    aggregated_funding_types_df = aggregate_by_funding_round_types2(funding_rounds_df.assign(investment_type = lambda df: df.investment_stage), period="year", min_year=2014, max_year=2025)
    aggregated_funding_types_startup_df = aggregate_by_funding_round_types2(funding_rounds_startup_df.assign(investment_type = lambda df: df.investment_stage), period="year", min_year=2014, max_year=2025)
    aggregated_funding_types_quarterly_df = aggregate_by_funding_round_types2(funding_rounds_df.assign(investment_type = lambda df: df.investment_stage), period="quarter", min_year=2023, max_year=2025)
    aggregated_funding_types_quarterly_startup_df = aggregate_by_funding_round_types2(funding_rounds_startup_df.assign(investment_type = lambda df: df.investment_stage), period="quarter", min_year=2023, max_year=2025)


    # IPOs and acquisitions
    ipos_df = CB.ipos.query("org_id in @matching_ids")
    acquisitions_df = CB.acquisitions.query("acquiree_id in @matching_ids")   

    # Figure variables
    if prefix is None:
        prefix = f"{OUTPUT_DIR}/charts/{category_name}_"
    _scale = 2

    # Chart: Investment amounts (total investment, quarterly)
    fig = charts.ts_bar(
        ts_quarterly_df,
        variable='raised_amount_gbp_total',
        variable_title="Raised amount, £ millions",
        category_column="_category",
        time_column="quarter"
    )
    fig = charts.configure_plots(fig, chart_title=f"Funding raised over time for {category_name}")
    chart_filename = f"{prefix}raised_amount_quarterly.png"
    fig.save(chart_filename, scale_factor=_scale)

    fig = charts.ts_bar(
        ts_df,
        variable='raised_amount_gbp_total',
        variable_title="Raised amount, £ millions",
        category_column="_category",
    )
    fig = charts.configure_plots(fig, chart_title=f"Funding raised over time for {category_name}")
    chart_filename = f"{prefix}raised_amount.png"
    fig.save(chart_filename, scale_factor=_scale)

    ## 
    fig = charts.ts_bar(
        ts_quarterly_startup_df,
        variable='raised_amount_gbp_total',
        variable_title="Raised amount, £ millions",
        category_column="_category",
        time_column="quarter"
    )
    fig = charts.configure_plots(fig, chart_title=f"Funding raised over time for {category_name} (early and growth stage)")
    chart_filename = f"{prefix}raised_amount_quarterly_startup.png"
    fig.save(chart_filename, scale_factor=_scale)

    fig = charts.ts_bar(
        ts_startup_df,
        variable='raised_amount_gbp_total',
        variable_title="Raised amount, £ millions",
        category_column="_category",
    )
    fig = charts.configure_plots(fig, chart_title=f"Funding raised over time for {category_name} (early and growth stage)")
    chart_filename = f"{prefix}raised_amount_startup.png"
    fig.save(chart_filename, scale_factor=_scale)

    ## ALL INVESTMENT - YEARLY
    # Investment amounts (by type)
    investment_types_fig = chart_investment_types(aggregated_funding_types_df, time_period_col="year")
    investment_types_fig = charts.configure_plots(investment_types_fig, chart_title=f"Breakdown of investment types for {category_name}")
    investment_types_chart_filename = f"{prefix}investment_types.png"
    investment_types_fig.save(investment_types_chart_filename, scale_factor=_scale)

    # Investment counts (by type)
    investment_types_counts_fig = chart_investment_types_counts(aggregated_funding_types_df, time_period_col="year")
    investment_types_counts_fig = charts.configure_plots(investment_types_counts_fig, chart_title=f"Number of investments by type for {category_name}")
    investment_types_counts_chart_filename = f"{prefix}investment_types_counts.png"
    investment_types_counts_fig.save(investment_types_counts_chart_filename, scale_factor=_scale)

    ## ALL INVESTMENT - QUARTERLY
    # Investment amounts (by type)
    investment_types_fig = chart_investment_types(aggregated_funding_types_quarterly_df, time_period_col="quarter")
    investment_types_fig = charts.configure_plots(investment_types_fig, chart_title=f"Breakdown of investment types for {category_name}")
    investment_types_chart_filename = f"{prefix}quarterly_investment_types.png"
    investment_types_fig.save(investment_types_chart_filename, scale_factor=_scale)

    # Investment counts (by type)
    investment_types_counts_fig = chart_investment_types_counts(aggregated_funding_types_quarterly_df, time_period_col="quarter")
    investment_types_counts_fig = charts.configure_plots(investment_types_counts_fig, chart_title=f"Number of investments by type for {category_name}")
    investment_types_counts_chart_filename = f"{prefix}quarterly_investment_types_counts.png"
    investment_types_counts_fig.save(investment_types_counts_chart_filename, scale_factor=_scale)

    ### EARLY AND GROWTH STAGE - YEARLY
    # Investment amounts (by type)
    investment_types_fig = chart_investment_types(aggregated_funding_types_startup_df)
    investment_types_fig = charts.configure_plots(investment_types_fig, chart_title=f"Breakdown of investment types for {category_name}")
    investment_types_chart_filename = f"{prefix}investment_types_startup.png"
    investment_types_fig.save(investment_types_chart_filename, scale_factor=_scale)

    # Investment counts (by type)
    investment_types_counts_fig = chart_investment_types_counts(aggregated_funding_types_startup_df)
    investment_types_counts_fig = charts.configure_plots(investment_types_counts_fig, chart_title=f"Number of investments by type for {category_name}")
    investment_types_counts_chart_filename = f"{prefix}investment_types_startup_counts.png"
    investment_types_counts_fig.save(investment_types_counts_chart_filename, scale_factor=_scale)

    ### EARLY AND GROWTH STAGE - QUARTERLY
    # Investment amounts (by type)
    investment_types_fig = chart_investment_types(aggregated_funding_types_quarterly_startup_df, time_period_col="quarter")
    investment_types_fig = charts.configure_plots(investment_types_fig, chart_title=f"Breakdown of investment types for {category_name}")
    investment_types_chart_filename = f"{prefix}quarterly_investment_types_startup.png"
    investment_types_fig.save(investment_types_chart_filename, scale_factor=_scale)

    # Investment counts (by type)
    investment_types_counts_fig = chart_investment_types_counts(aggregated_funding_types_quarterly_startup_df, time_period_col="quarter")
    investment_types_counts_fig = charts.configure_plots(investment_types_counts_fig, chart_title=f"Number of investments by type for {category_name}")
    investment_types_counts_chart_filename = f"{prefix}quarterly_investment_types_startup_counts.png"
    investment_types_counts_fig.save(investment_types_counts_chart_filename, scale_factor=_scale)

    return (
        ts_df,
        ts_quarterly_df,
        ts_quarterly_startup_df,
        growth_magnitude_df,
        growth_magnitude_quarterly_df,
        ipos_df,
        acquisitions_df,
        matchings_orgs_df,
        funding_rounds_df,
        aggregated_funding_types_df,
        aggregated_funding_types_startup_df,
        aggregated_funding_types_quarterly_df,
        aggregated_funding_types_quarterly_startup_df
    )

In [11]:
# (
#     funding_rounds_df
#     .query("year >= 2024")
#     .query("investment_stage == 'other'")
#     .sort_values("raised_amount_gbp", ascending=False)
#     [['cb_url', "funding_round_name", "raised_amount_gbp", "investor_name"]]
# )

## Run all categories

In [12]:
# funding_round_types = FUNDING_ROUND_TYPES
# prefix = f"{OUTPUT_DIR}/charts/{category_name}_"
# table_prefix = f"{OUTPUT_DIR}"

funding_round_types = None
prefix = None
table_prefix = f"{OUTPUT_DIR}/cb_"

In [13]:
all_ts_df = []
all_ts_quarterly_df = []
all_ts_quarterly_startup_df = []
all_growth_magnitude_df = []
all_growth_magnitude_quarterly_df = []
all_ipos_df = []
all_acquisitions_df = []
all_orgs = []
all_funding_rounds_df = []

all_aggregated_funding_types_df = []
all_aggregated_funding_types_startup_df = []
all_aggregated_funding_types_quarterly_df = []
all_aggregated_funding_types_quarterly_startup_df = []


In [28]:
for config_name in src_utils.CONFIG_NAMES:
    logging.info(f"Processing {config_name}")
    config = src_utils.get_config_dict(config_name)
    category_name = config["search_recipe"]["category_name"]
    matching_ids = get_ids_from_config(config_name)
    
    ts_df, ts_quarterly_df, ts_quarterly_startup_df, growth_magnitude_df, growth_magnitude_quarterly_df, ipos_df, acquisitions_df, matchings_orgs_df, funding_rounds_df, aggregated_funding_types_df,aggregated_funding_types_startup_df,aggregated_funding_types_quarterly_df,aggregated_funding_types_quarterly_startup_df = produce_stats(CB, matching_ids, category_name, funding_round_types, prefix)
   
    all_ts_df.append(ts_df.assign(theme=category_name))
    all_ts_quarterly_df.append(ts_quarterly_df.assign(theme=category_name))
    all_ts_quarterly_startup_df.append(ts_quarterly_startup_df.assign(theme=category_name))
    all_growth_magnitude_df.append(growth_magnitude_df)
    all_growth_magnitude_quarterly_df.append(growth_magnitude_quarterly_df)
    all_ipos_df.append(ipos_df.assign(theme=category_name))
    all_acquisitions_df.append(acquisitions_df.assign(theme=category_name))
    all_orgs.append(matchings_orgs_df.assign(theme=category_name))
    all_funding_rounds_df.append(funding_rounds_df.assign(theme=category_name))

    all_aggregated_funding_types_df.append(aggregated_funding_types_df.assign(theme=category_name))
    all_aggregated_funding_types_startup_df.append(aggregated_funding_types_startup_df.assign(theme=category_name))
    all_aggregated_funding_types_quarterly_df.append(aggregated_funding_types_quarterly_df.assign(theme=category_name))
    all_aggregated_funding_types_quarterly_startup_df.append(aggregated_funding_types_quarterly_startup_df.assign(theme=category_name))
    

2025-04-14 10:21:02,994 - root - INFO - Processing biomass_heating
2025-04-14 10:21:05,748 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/enriched/funding_rounds_full.parquet
2025-04-14 10:21:05,932 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-14 10:21:05,939 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-14 10:21:05,942 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-14 10:21:05,949 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-14 10:21:05,954 - botocore.httpchecksum - INFO - Skipping checksum validation. R

In [29]:
for config_name, category in src_utils.CB_CATEGORIES.items():
    logging.info(f"Processing {config_name}")
    config = src_utils.get_config_dict(config_name)
    category_name = config["search_recipe"]["category_name"]
    matching_ids = CB.get_companies_in_categories([category], "narrow").id.to_list()
    
    ts_df, ts_quarterly_df, ts_quarterly_startup_df, growth_magnitude_df, growth_magnitude_quarterly_df, ipos_df, acquisitions_df, matchings_orgs_df, funding_rounds_df, aggregated_funding_types_df,aggregated_funding_types_startup_df,aggregated_funding_types_quarterly_df,aggregated_funding_types_quarterly_startup_df = produce_stats(CB, matching_ids, category_name, funding_round_types, prefix)
   
    all_ts_df.append(ts_df.assign(theme=category_name))
    all_ts_quarterly_df.append(ts_quarterly_df.assign(theme=category_name))
    all_ts_quarterly_startup_df.append(ts_quarterly_startup_df.assign(theme=category_name))
    all_growth_magnitude_df.append(growth_magnitude_df)
    all_growth_magnitude_quarterly_df.append(growth_magnitude_quarterly_df)
    all_ipos_df.append(ipos_df.assign(theme=category_name))
    all_acquisitions_df.append(acquisitions_df.assign(theme=category_name))
    all_orgs.append(matchings_orgs_df.assign(theme=category_name))
    all_funding_rounds_df.append(funding_rounds_df.assign(theme=category_name))

    all_aggregated_funding_types_df.append(aggregated_funding_types_df.assign(theme=category_name))
    all_aggregated_funding_types_startup_df.append(aggregated_funding_types_startup_df.assign(theme=category_name))
    all_aggregated_funding_types_quarterly_df.append(aggregated_funding_types_quarterly_df.assign(theme=category_name))
    all_aggregated_funding_types_quarterly_startup_df.append(aggregated_funding_types_quarterly_startup_df.assign(theme=category_name))

2025-04-14 10:21:56,078 - root - INFO - Processing energy_storage
2025-04-14 10:22:08,770 - root - INFO - Processing renewables_general
2025-04-14 10:22:18,912 - root - INFO - Processing solar
2025-04-14 10:22:26,325 - root - INFO - Processing wind


## Low carbon heating - standalone

In [30]:
category_name = "Low-Carbon Heating"
ts_df, ts_quarterly_df, ts_quarterly_startup_df, growth_magnitude_df, growth_magnitude_quarterly_df, ipos_df, acquisitions_df, matchings_orgs_df, funding_rounds_df, aggregated_funding_types_df,aggregated_funding_types_startup_df,aggregated_funding_types_quarterly_df,aggregated_funding_types_quarterly_startup_df = produce_stats(CB, lch_ids, category_name, funding_round_types, prefix)

all_ts_df.append(ts_df.assign(theme=category_name))
all_ts_quarterly_df.append(ts_quarterly_df.assign(theme=category_name))
all_ts_quarterly_startup_df.append(ts_quarterly_startup_df.assign(theme=category_name))
all_growth_magnitude_df.append(growth_magnitude_df)
all_growth_magnitude_quarterly_df.append(growth_magnitude_quarterly_df)
all_ipos_df.append(ipos_df.assign(theme=category_name))
all_acquisitions_df.append(acquisitions_df.assign(theme=category_name))
all_orgs.append(matchings_orgs_df.assign(theme=category_name))
all_funding_rounds_df.append(funding_rounds_df.assign(theme=category_name))

all_aggregated_funding_types_df.append(aggregated_funding_types_df.assign(theme=category_name))
all_aggregated_funding_types_startup_df.append(aggregated_funding_types_startup_df.assign(theme=category_name))
all_aggregated_funding_types_quarterly_df.append(aggregated_funding_types_quarterly_df.assign(theme=category_name))
all_aggregated_funding_types_quarterly_startup_df.append(aggregated_funding_types_quarterly_startup_df.assign(theme=category_name))


In [31]:
all_ts_df = pd.concat(all_ts_df, ignore_index=True)
all_ts_df.to_csv(f"{table_prefix}all_ts_df.csv", index=False)

all_ts_quarterly_df = pd.concat(all_ts_quarterly_df, ignore_index=True)
all_ts_quarterly_df.to_csv(f"{table_prefix}all_ts_quarterly_df.csv", index=False)

all_ts_quarterly_startup_df = pd.concat(all_ts_quarterly_startup_df, ignore_index=True)
all_ts_quarterly_startup_df.to_csv(f"{table_prefix}all_ts_quarterly_startup_df.csv", index=False)

all_growth_magnitude_df = pd.concat(all_growth_magnitude_df, ignore_index=True)
all_growth_magnitude_df.to_csv(f"{table_prefix}growth_magnitude.csv", index=False)

all_growth_magnitude_quarterly_df = pd.concat(all_growth_magnitude_quarterly_df, ignore_index=True)
all_growth_magnitude_quarterly_df.to_csv(f"{table_prefix}growth_magnitude_quarterly.csv", index=False)

try:
    all_ipos_df = pd.concat(all_ipos_df, ignore_index=True).to_csv(f"{table_prefix}all_ipos.csv", index=False)
    all_acquisitions_df = pd.concat(all_acquisitions_df, ignore_index=True).to_csv(f"{table_prefix}all_acquisitions.csv", index=False)
except:
    pass

all_orgs = pd.concat(all_orgs, ignore_index=True)
all_orgs.to_csv(f"{table_prefix}all_orgs.csv", index=False)

all_funding_rounds_df = pd.concat(all_funding_rounds_df, ignore_index=True)
all_funding_rounds_df.to_csv(f"{table_prefix}all_funding_rounds.csv", index=False)

all_aggregated_funding_types_df = pd.concat(all_aggregated_funding_types_df, ignore_index=True)
all_aggregated_funding_types_df.to_csv(f"{table_prefix}all_aggregated_funding_types.csv", index=False)

all_aggregated_funding_types_startup_df = pd.concat(all_aggregated_funding_types_startup_df, ignore_index=True)
all_aggregated_funding_types_startup_df.to_csv(f"{table_prefix}all_aggregated_funding_types_startup.csv", index=False)

all_aggregated_funding_types_quarterly_df = pd.concat(all_aggregated_funding_types_quarterly_df, ignore_index=True)
all_aggregated_funding_types_quarterly_df.to_csv(f"{table_prefix}all_aggregated_funding_types_quarterly.csv", index=False)

all_aggregated_funding_types_quarterly_startup_df = pd.concat(all_aggregated_funding_types_quarterly_startup_df, ignore_index=True)
all_aggregated_funding_types_quarterly_startup_df.to_csv(f"{table_prefix}all_aggregated_funding_types_quarterly_startup.csv", index=False)



In [32]:
(
    all_funding_rounds_df
    .query("year > 2023 and year < 2025")
    .drop_duplicates(["funding_round_id"])
    .query("investment_type == 'seed'")
    .assign(dummy_col="include")
    .dropna(subset=["raised_amount_gbp"])
    .groupby(["dummy_col"])
    .agg(
        amount_mean=("raised_amount_gbp", "mean"),
        amount_median=("raised_amount_gbp", "median"),
    )
)

,amount_mean,amount_median
dummy_col,,
include,4111.168385,1817.152104


In [33]:
(
    all_funding_rounds_df
    .query("year > 2023 and year < 2025")
    .drop_duplicates(["funding_round_id"])
    .query("investment_type == 'series_a'")
    .assign(dummy_col="include")
    .dropna(subset=["raised_amount_gbp"])
    .groupby(["dummy_col"])
    .agg(
        amount_mean=("raised_amount_gbp", "mean"),
        amount_median=("raised_amount_gbp", "median"),
    )
)

,amount_mean,amount_median
dummy_col,,
include,19424.546037,9325.386


In [34]:
(
    all_funding_rounds_df
    .query("year > 2023 and year < 2025")
    .drop_duplicates(["funding_round_id"])
    .query("investment_type == 'debt_financing'")
    .assign(dummy_col="include")
    .dropna(subset=["raised_amount_gbp"])
    .groupby(["dummy_col"])
    .agg(
        amount_mean=("raised_amount_gbp", "mean"),
        amount_median=("raised_amount_gbp", "median"),
    )
)

,amount_mean,amount_median
dummy_col,,
include,346481.345726,85795.0


In [38]:
## Baseline

In [35]:
matching_ids = CB.organisations_enriched.id.tolist()
ts_df, ts_quarterly_df, ts_quarterly_startup_df, growth_magnitude_df, growth_magnitude_quarterly_df, _, _, _, _, _, _, _, _ = produce_stats(CB, matching_ids, category_name="All investment (baseline)", funding_round_types=None, prefix=None)

In [36]:
growth_magnitude_df

,variable,magnitude,growth,theme
0,n_rounds,37881.600000,-2.002120,All investment (baseline)
1,raised_amount_usd_total,321449.827465,10.437503,All investment (baseline)
2,raised_amount_gbp_total,246990.038311,13.994380,All investment (baseline)
3,n_orgs_founded,51228.800000,-68.517777,All investment (baseline)


In [37]:
growth_magnitude_quarterly_df

_col,variable,previous_four_quarters,magnitude,growth,theme
0,raised_amount_gbp_total,47085.591536,42468.104912,-9.806581,All investment (baseline)
1,n_rounds,6860.500000,4683.000000,-31.739669,All investment (baseline)
2,n_orgs_founded,4348.000000,1641.000000,-62.258510,All investment (baseline)
